In [15]:
import urllib.request, zipfile

url = "http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip"
urllib.request.urlretrieve(url, "cornell.zip")

with zipfile.ZipFile("cornell.zip", 'r') as zip_ref:
    zip_ref.extractall()

print("Dataset ready ✅")

Dataset ready ✅


In [16]:
lines = open("cornell movie-dialogs corpus/movie_lines.txt",
             encoding='utf-8', errors='ignore').read().split('\n')

id2line = {}

for line in lines:
    parts = line.split(" +++$+++ ")
    if len(parts) == 5:
        id2line[parts[0]] = parts[4]

print("Total lines:", len(id2line))

Total lines: 304713


In [17]:
convs = open("cornell movie-dialogs corpus/movie_conversations.txt",
             encoding='utf-8', errors='ignore').read().split('\n')

conversations = []

for conv in convs:
    parts = conv.split(" +++$+++ ")
    if len(parts) == 4:
        line_ids = eval(parts[3])  # list of IDs
        conversations.append(line_ids)

print("Total conversations:", len(conversations))


Total conversations: 83097


In [18]:
pairs = []

for conv in conversations:
    for i in range(len(conv) - 1):
        input_line = id2line.get(conv[i], "")
        target_line = id2line.get(conv[i+1], "")

        if input_line and target_line:
            pairs.append((input_line, target_line))

print("Sample pairs:")
print(pairs[:5])

Sample pairs:
[('Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.', "Well, I thought we'd start with pronunciation, if that's okay with you."), ("Well, I thought we'd start with pronunciation, if that's okay with you.", 'Not the hacking and gagging and spitting part.  Please.'), ('Not the hacking and gagging and spitting part.  Please.', "Okay... then how 'bout we try out some French cuisine.  Saturday?  Night?"), ("You're asking me out.  That's so cute. What's your name again?", 'Forget it.'), ("No, no, it's my fault -- we didn't have a proper introduction ---", 'Cameron.')]


In [19]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9]+", " ", text)
    return text.strip()

pairs_cleaned = [(clean_text(q), clean_text(a)) for q,a in pairs]

pairs_cleaned = [(q,a) for q,a in pairs_cleaned if q and a]

print(pairs_cleaned[:5])

[('can we make this quick roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad again', 'well i thought we d start with pronunciation if that s okay with you'), ('well i thought we d start with pronunciation if that s okay with you', 'not the hacking and gagging and spitting part please'), ('not the hacking and gagging and spitting part please', 'okay then how bout we try out some french cuisine saturday night'), ('you re asking me out that s so cute what s your name again', 'forget it'), ('no no it s my fault we didn t have a proper introduction', 'cameron')]


In [20]:
word2idx = {"<PAD>":0, "<SOS>":1, "<EOS>":2}
idx2word = {0:"<PAD>", 1:"<SOS>", 2:"<EOS>"}

def build_vocab(pairs):
    idx = 3
    for q,a in pairs:
        for word in q.split() + a.split():
            if word not in word2idx:
                word2idx[word] = idx
                idx2word[idx] = word
                idx += 1

build_vocab(pairs_cleaned[:10000])  # limit for speed

vocab_size = len(word2idx)
print("Vocab size:", vocab_size)

Vocab size: 10215


In [21]:
import torch

def encode(sentence):
    return [word2idx.get(word, 0) for word in sentence.split()]

def tensorize(pair):
    q, a = pair
    return (
        torch.tensor([1] + encode(q) + [2]),
        torch.tensor([1] + encode(a) + [2])
    )

dataset = [tensorize(p) for p in pairs_cleaned[:10000]]

In [22]:
class ChatDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    inputs, targets = zip(*batch)
    inputs = nn.utils.rnn.pad_sequence(inputs, batch_first=True)
    targets = nn.utils.rnn.pad_sequence(targets, batch_first=True)
    return inputs, targets

loader = DataLoader(ChatDataset(dataset), batch_size=32, shuffle=True, collate_fn=collate_fn)

In [23]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim*2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

In [24]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

In [25]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = Attention(hidden_dim)
        self.lstm = nn.LSTM(hidden_dim + embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)
        embedded = self.embedding(x)

        attn_weights = self.attention(hidden[-1], encoder_outputs)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)

        lstm_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

In [26]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(src.device)

        encoder_outputs, hidden, cell = self.encoder(src)

        x = trg[:,0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(x, hidden, cell, encoder_outputs)
            outputs[:,t] = output
            x = trg[:,t]

        return outputs

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(vocab_size, 128, 256)
decoder = Decoder(vocab_size, 128, 256)

model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [28]:
for epoch in range(5):
    model.train()
    total_loss = 0

    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()
        output = model(src, trg)

        output_dim = output.shape[-1]
        output = output[:,1:].reshape(-1, output_dim)
        trg = trg[:,1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader)}")


Epoch 1, Loss: 6.081511962147186
Epoch 2, Loss: 5.41458269734733
Epoch 3, Loss: 5.139242419038718
Epoch 4, Loss: 4.926844068228627
Epoch 5, Loss: 4.738255573918644


In [29]:
def generate_response(sentence):
    model.eval()

    sentence = clean_text(sentence)
    tensor = torch.tensor([1] + encode(sentence) + [2]).unsqueeze(0).to(device)

    encoder_outputs, hidden, cell = model.encoder(tensor)

    x = torch.tensor([1]).to(device)
    result = []

    for _ in range(20):
        output, hidden, cell = model.decoder(x, hidden, cell, encoder_outputs)
        pred = output.argmax(1).item()

        if pred == 2:
            break

        result.append(idx2word.get(pred, ""))
        x = torch.tensor([pred]).to(device)

    return " ".join(result)

print(generate_response("how are you"))

i m not a good
